# Region-reactive: compute coverage only for what's on screen

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GMOD/jbrowse-anywidget/blob/main/examples/10_region_reactive.ipynb)

The view syncs its visible region back to Python, so you can **observe `location` and recompute as the user pans** — the loop a static browser can't close. Here [pysam](https://pysam.readthedocs.io) counts coverage from the real NA12878 exome BAM only over the window in view, at a bin size that follows the zoom. Nothing is precomputed genome-wide; the kernel answers for exactly what's asked, so zooming in *raises* the resolution instead of cropping a fixed file.

In [ ]:
# Install only if not already available (e.g. in Colab). The GitHub install
# needs no JS toolchain — the built widget bundle is committed in the repo. A
# local editable install is used as-is. (Swap to `jbrowse-anywidget` once it's
# published to PyPI.)
try:
    import jbrowse_anywidget  # noqa: F401
except ImportError:
    %pip install -q "jbrowse-anywidget @ git+https://github.com/GMOD/jbrowse-anywidget" pandas numpy pysam

# Colab requires this to render third-party (anywidget) widgets:
try:
    from google.colab import output

    output.enable_custom_widget_manager()
except ImportError:
    pass

## Coverage for one window

Open the remote BAM once — only its index and the regions you query are fetched. `coverage` counts per-base depth over `start..end` and bins to ~400 points across the view. The BAM names the chromosome `17`; the hg19 hub uses `chr17`, so we strip the prefix on the way in.

In [ ]:
import numpy as np
import pandas as pd
import pysam

BAM = (
    "https://s3.amazonaws.com/1000genomes/phase3/data/NA12878/"
    "exome_alignment/NA12878.mapped.ILLUMINA.bwa.CEU.exome.20121211.bam"
)
bam = pysam.AlignmentFile(BAM)


def coverage(chrom, start, end):
    depth = np.array(bam.count_coverage(chrom.removeprefix("chr"), start, end)).sum(0)
    binsize = max(20, (end - start) // 400)
    n = depth.size // binsize * binsize
    binned = depth[:n].reshape(-1, binsize).mean(1).round(1)
    starts = start + np.arange(binned.size) * binsize
    return pd.DataFrame(
        {"chrom": chrom, "start": starts, "end": starts + binsize, "depth": binned}
    )


coverage("chr17", 41_196_312, 41_277_500).head()  # BRCA1

## Recompute on every pan

`on_location` parses the view's locstring and re-renders coverage for that window. `view.observe(..., "location")` fires it whenever the region changes — dragging in the UI or setting `view.location` from code. A gene-name or whole-chromosome location doesn't parse, and a window wider than 5 Mb is skipped to keep each per-pan query snappy.

In [ ]:
import re

from jbrowse_anywidget import LinearGenomeView, fetch_hub

hg19 = fetch_hub("hg19")
COLOR = "jexl:get(feature,'depth') > 40 ? '#c62828' : get(feature,'depth') > 10 ? '#f9a825' : '#cfcfcf'"


def parse_loc(loc):
    m = re.match(r"^\s*([^:\s]+)\s*:\s*([\d,]+)\s*\.\.\s*([\d,]+)", loc or "")
    return (m[1], int(m[2].replace(",", "")), int(m[3].replace(",", ""))) if m else None


def render_region(chrom, start, end):
    if end - start <= 5_000_000:
        view.tracks = []  # replace with the freshly computed window
        view.add_features(
            coverage(chrom, start, end),
            name="NA12878 exome depth (visible region)",
            track_id="depth",
            color=COLOR,
        )


def on_location(change):
    region = parse_loc(change["new"])
    if region:
        render_region(*region)


view = LinearGenomeView(
    assembly=hg19["assemblies"][0],
    aggregate_text_search_adapters=hg19["aggregateTextSearchAdapters"],
    location="BRCA1",
)
view.observe(on_location, "location")
view  # pan or zoom — the depth track recomputes for the new window

Driving `location` from code fires the same observer, so the track recomputes for the new window. Zooming out widens the bins; zooming in sharpens them — the resolution follows the view:

In [ ]:
view.location = "chr17:7,560,000..7,595,000"  # jump to TP53
len(view.tracks[0]["adapter"]["features"]), "bins computed for this window"